In [1]:
import pandas as pd
import numpy as np
import os

# We always load from raw/ — never from processed/
# This ensures our cleaning notebook is fully reproducible
# Anyone can run this notebook from scratch and get the same result

file_path = os.path.join('..', 'data', 'raw', 'merged_jee_cutoff_2018_2025.csv')
df_raw = pd.read_csv(file_path)

print(f"Raw data loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

Raw data loaded: 433993 rows, 9 columns


In [2]:
# IMPORTANT PROFESSIONAL HABIT:
# We never modify df_raw directly
# We create a copy called df_clean and apply all changes to it
# This means if something goes wrong, df_raw is always intact in memory
# .copy() creates a completely independent copy — not just a reference

df_clean = df_raw.copy()

print("Working copy created.")
print(f"df_raw rows    : {len(df_raw)}")
print(f"df_clean rows  : {len(df_clean)}")
print(f"Are they the same object? {df_raw is df_clean}")

Working copy created.
df_raw rows    : 433993
df_clean rows  : 433993
Are they the same object? False


In [3]:
# Current column names have spaces and mixed cases
# Example: 'Academic Program Name', 'Opening Rank'
# In analysis we reference columns hundreds of times
# snake_case (lowercase with underscores) is the industry standard
# It prevents errors from accidental capitalization mismatches

df_clean.columns = [
    'institute',
    'program',
    'quota',
    'seat_type',
    'gender',
    'opening_rank',
    'closing_rank',
    'round',
    'year'
]

print("Columns renamed to snake_case:")
print(df_clean.columns.tolist())

Columns renamed to snake_case:
['institute', 'program', 'quota', 'seat_type', 'gender', 'opening_rank', 'closing_rank', 'round', 'year']


In [5]:
# Problem identified in Phase 3:
# "Birla Institute of Technology, Mesra,  Ranchi"  ← extra space
# "Birla Institute of Technology, Mesra, Ranchi"   ← correct
# "Indian Institute  of Technology (BHU) Varanasi" ← double space
#
# .str.strip() removes leading and trailing spaces
# .str.replace() with regex=True replaces multiple internal spaces with one
# We apply this to both institute and program columns

before = df_clean['institute'].nunique()

df_clean['institute'] = (
    df_clean['institute']
    .str.strip()                          # Remove spaces at start and end
    .str.replace(r'\s+', ' ', regex=True) # Replace multiple spaces with single space
)

df_clean['program'] = (
    df_clean['program']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

after = df_clean['institute'].nunique()

print(f"Unique institutes before cleaning : {before}")
print(f"Unique institutes after cleaning  : {after}")
print(f"Duplicates removed                : {before - after}")

Unique institutes before cleaning : 137
Unique institutes after cleaning  : 137
Duplicates removed                : 0


In [6]:
# Problem: 1 row has value 'F' instead of full gender label
# We replace it with the correct full value
# .replace() on a Series replaces exact matches only

before = df_clean['gender'].value_counts()
print("Before fix:")
print(before)

df_clean['gender'] = df_clean['gender'].replace(
    'F', 
    'Female-only (including Supernumerary)'
)

after = df_clean['gender'].value_counts()
print("\nAfter fix:")
print(after)

print(f"\nVerification - unique gender values now: {df_clean['gender'].nunique()}")

Before fix:
gender
Gender-Neutral                           272984
Female-only (including Supernumerary)    161005
F                                             1
Name: count, dtype: int64

After fix:
gender
Gender-Neutral                           272984
Female-only (including Supernumerary)    161006
Name: count, dtype: int64

Verification - unique gender values now: 2


In [7]:
# GO appeared in Quota column with 1,285 rows
# Before we decide what to do, we need to see which institutes use it
# and which years it appears in

go_rows = df_clean[df_clean['quota'] == 'GO']

print(f"Total GO quota rows: {len(go_rows)}")
print(f"\nYears where GO appears:")
print(go_rows['year'].value_counts().sort_index())
print(f"\nInstitutes using GO quota:")
print(go_rows['institute'].unique())

Total GO quota rows: 1285

Years where GO appears:
year
2018    186
2019    209
2020    204
2021    104
2022    202
2023    181
2025    199
Name: count, dtype: int64

Institutes using GO quota:
<StringArray>
['National Institute of Technology Goa']
Length: 1, dtype: str


In [8]:
# Round 7 appeared only in 2018 and 2019
# We need to understand what it represents before deciding

round7_rows = df_clean[df_clean['round'] == 7]

print(f"Total Round 7 rows: {len(round7_rows)}")
print(f"\nYears where Round 7 appears:")
print(round7_rows['year'].value_counts())
print(f"\nSample institutes in Round 7:")
print(round7_rows['institute'].unique()[:10])
print(f"\nSeat types in Round 7:")
print(round7_rows['seat_type'].value_counts())

Total Round 7 rows: 14753

Years where Round 7 appears:
year
2019    8051
2018    6702
Name: count, dtype: int64

Sample institutes in Round 7:
<StringArray>
['Indian Institute of Technology Bhubaneswar',
      'Indian Institute of Technology Bombay',
       'Indian Institute of Technology Mandi',
       'Indian Institute of Technology Delhi',
      'Indian Institute of Technology Indore',
   'Indian Institute of Technology Kharagpur',
   'Indian Institute of Technology Hyderabad',
     'Indian Institute of Technology Jodhpur',
      'Indian Institute of Technology Kanpur',
      'Indian Institute of Technology Madras']
Length: 10, dtype: str

Seat types in Round 7:
seat_type
OPEN             3446
OBC-NCL          3127
SC               3105
ST               2507
EWS              1121
OPEN (PwD)        927
OBC-NCL (PwD)     359
SC (PwD)           87
ST (PwD)           38
EWS (PwD)          36
Name: count, dtype: int64


In [9]:
# Maximum rank was 1,368,129 which exceeds JEE Main range
# Let us understand how many such rows exist and what they look like

# JEE Main total candidates approximately 1,200,000
# We flag anything above this threshold for investigation

high_rank_threshold = 1200000

high_ranks = df_clean[
    (df_clean['closing_rank'] > high_rank_threshold) | 
    (df_clean['opening_rank'] > high_rank_threshold)
]

print(f"Rows with rank above {high_rank_threshold:,}: {len(high_ranks)}")
print(f"\nYears these appear in:")
print(high_ranks['year'].value_counts().sort_index())
print(f"\nSeat types for high rank rows:")
print(high_ranks['seat_type'].value_counts())
print(f"\nSample rows:")
print(high_ranks[['institute', 'program', 'seat_type', 
                   'opening_rank', 'closing_rank', 'year']].head(10))

Rows with rank above 1,200,000: 15

Years these appear in:
year
2024     5
2025    10
Name: count, dtype: int64

Seat types for high rank rows:
seat_type
OPEN    15
Name: count, dtype: int64

Sample rows:
                                        institute  \
283925    National Institute of Technology Sikkim   
283926    National Institute of Technology Sikkim   
288340  National Institute of Technology, Mizoram   
288490  National Institute of Technology, Mizoram   
288491  National Institute of Technology, Mizoram   
369551  National Institute of Technology, Mizoram   
381584  National Institute of Technology, Mizoram   
381636  National Institute of Technology, Mizoram   
381641  National Institute of Technology, Mizoram   
393504  National Institute of Technology, Mizoram   

                                                  program seat_type  \
283925  Civil Engineering (4 Years, Bachelor of Techno...      OPEN   
283926  Civil Engineering (4 Years, Bachelor of Techno...      OPEN  

In [10]:
# We found ~1,468 rows with missing Opening Rank
# and ~1,469 rows with missing Closing Rank
# Before we decide to drop or fill, we need to understand the pattern
# Are they missing randomly? Or only for specific years/categories?

missing_opening = df_clean[df_clean['opening_rank'].isna()]
missing_closing = df_clean[df_clean['closing_rank'].isna()]

print(f"Missing Opening Rank: {len(missing_opening)}")
print(f"Missing Closing Rank: {len(missing_closing)}")

print(f"\nMissing Opening Rank by Year:")
print(missing_opening['year'].value_counts().sort_index())

print(f"\nMissing Opening Rank by Seat Type:")
print(missing_opening['seat_type'].value_counts())

print(f"\nMissing Opening Rank by Round:")
print(missing_opening['round'].value_counts().sort_index())

# Check if missing opening and closing ranks are in the same rows
both_missing = df_clean[
    df_clean['opening_rank'].isna() & 
    df_clean['closing_rank'].isna()
]
print(f"\nRows where BOTH ranks are missing: {len(both_missing)}")

Missing Opening Rank: 1468
Missing Closing Rank: 1469

Missing Opening Rank by Year:
year
2019       1
2021       3
2024    1464
Name: count, dtype: int64

Missing Opening Rank by Seat Type:
seat_type
OPEN (PwD)       369
OBC-NCL (PwD)    357
SC (PwD)         204
ST               132
EWS (PwD)        126
OBC-NCL          104
EWS               98
ST (PwD)          47
SC                17
OPEN              12
Name: count, dtype: int64

Missing Opening Rank by Round:
round
1      1
2    296
3    367
4    393
5    410
6      1
Name: count, dtype: int64

Rows where BOTH ranks are missing: 1468


In [11]:
# Verify current column names before continuing
print(df_clean.columns.tolist())

['institute', 'program', 'quota', 'seat_type', 'gender', 'opening_rank', 'closing_rank', 'round', 'year']


In [12]:
# We keep all Round 7 rows but create a flag column
# This gives future analysts the choice to include or exclude special rounds
# without losing the data permanently

df_clean['is_special_round'] = df_clean['round'].apply(
    lambda x: True if x == 7 else False
)

print("Special round flag added.")
print(df_clean['is_special_round'].value_counts())

Special round flag added.
is_special_round
False    419240
True      14753
Name: count, dtype: int64


In [13]:
# We drop rows where BOTH opening_rank AND closing_rank are missing
# These represent seats that were never filled — no rank data exists
# We cannot impute ranks because there is no logical basis for estimation

rows_before = len(df_clean)

df_clean = df_clean.dropna(
    subset=['opening_rank', 'closing_rank'], 
    how='all'   # 'all' means drop only if BOTH are NaN
                # 'any' would drop if EITHER is NaN — too aggressive
)

rows_after = len(df_clean)

print(f"Rows before dropping  : {rows_before:,}")
print(f"Rows after dropping   : {rows_after:,}")
print(f"Rows removed          : {rows_before - rows_after:,}")
print(f"Percentage removed    : {((rows_before - rows_after)/rows_before*100):.2f}%")

Rows before dropping  : 433,993
Rows after dropping   : 432,525
Rows removed          : 1,468
Percentage removed    : 0.34%


In [22]:
# Now that zero NaN values remain, conversion to int is safe
df_clean['opening_rank'] = df_clean['opening_rank'].astype(int)
df_clean['closing_rank'] = df_clean['closing_rank'].astype(int)

print("Data types after conversion:")
print(df_clean[['opening_rank', 'closing_rank']].dtypes)
print(f"\nSample values:")
print(df_clean[['opening_rank', 'closing_rank']].head())

Data types after conversion:
opening_rank    int64
closing_rank    int64
dtype: object

Sample values:
   opening_rank  closing_rank
0          5057          6780
1         10078         10789
2          1649          2592
3          4343          4522
4          1054          1233


In [21]:
# One row has closing_rank missing but opening_rank present
# how='all' in Cell 11 only dropped rows where BOTH were missing
# This remaining row has no closing rank so it is unusable for analysis
# We drop it now using how='any' targeted only at remaining nulls

rows_before = len(df_clean)

df_clean = df_clean.dropna(subset=['opening_rank', 'closing_rank'], how='any')

rows_after = len(df_clean)

print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after}")

# Verify no missing ranks remain
print(f"\nMissing opening_rank : {df_clean['opening_rank'].isna().sum()}")
print(f"Missing closing_rank : {df_clean['closing_rank'].isna().sum()}")

Rows before : 432,525
Rows after  : 432,524
Rows removed: 1

Missing opening_rank : 0
Missing closing_rank : 0


In [15]:
# We already fixed spacing in Cell 4
# Now we standardize case consistency across all text columns
# .str.strip() removes any remaining edge spaces
# This is a final safety pass

text_cols = ['institute', 'program', 'quota', 'seat_type', 'gender']

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

print("Text columns stripped of edge whitespace.")

# Verify gender fix from earlier is still intact
print("\nGender unique values:")
print(df_clean['gender'].value_counts())

Text columns stripped of edge whitespace.

Gender unique values:
gender
Gender-Neutral                           271800
Female-only (including Supernumerary)    160725
Name: count, dtype: int64


In [16]:
# Our dataset has no Institute Type column
# But institute names follow consistent patterns we can use
# This is called FEATURE EXTRACTION from existing data
# 
# Pattern logic:
# "Indian Institute of Technology" → IIT
# "National Institute of Technology" → NIT  
# "Indian Institute of Information Technology" → IIIT
# Everything else → GFTI (Government Funded Technical Institute)
#
# We use .str.contains() to detect these patterns
# case=False makes it case-insensitive for safety

def classify_institute(name):
    if 'Indian Institute of Technology' in name:
        return 'IIT'
    elif 'National Institute of Technology' in name:
        return 'NIT'
    elif 'Indian Institute of Information Technology' in name:
        return 'IIIT'
    else:
        return 'GFTI'

df_clean['institute_type'] = df_clean['institute'].apply(classify_institute)

print("Institute type distribution:")
print(df_clean['institute_type'].value_counts())
print(f"\nSample mapping:")
print(df_clean[['institute', 'institute_type']].drop_duplicates().head(10))

Institute type distribution:
institute_type
NIT     233875
IIT     119011
GFTI     49186
IIIT     30453
Name: count, dtype: int64

Sample mapping:
                                      institute institute_type
0    Indian Institute of Technology Bhubaneswar            IIT
100       Indian Institute of Technology Bombay            IIT
252        Indian Institute of Technology Mandi            IIT
286        Indian Institute of Technology Delhi            IIT
417       Indian Institute of Technology Indore            IIT
458    Indian Institute of Technology Kharagpur            IIT
755    Indian Institute of Technology Hyderabad            IIT
824      Indian Institute of Technology Jodhpur            IIT
858       Indian Institute of Technology Kanpur            IIT
967       Indian Institute of Technology Madras            IIT


In [17]:
# Let us specifically check what got classified as GFTI
# to make sure no IIT or NIT was misclassified

gfti_institutes = (
    df_clean[df_clean['institute_type'] == 'GFTI']['institute']
    .unique()
)

print(f"Total GFTI institutes: {len(gfti_institutes)}")
print("\nAll GFTI institute names:")
for inst in sorted(gfti_institutes):
    print(f"  {inst}")

Total GFTI institutes: 57

All GFTI institute names:
  Assam University, Silchar
  Birla Institute of Technology, Deoghar Off-Campus
  Birla Institute of Technology, Mesra, Ranchi
  Birla Institute of Technology, Patna Off-Campus
  CU Jharkhand
  Central University of Haryana
  Central University of Jammu
  Central University of Rajasthan, Rajasthan
  Central institute of Technology Kokrajar, Assam
  Chhattisgarh Swami Vivekanada Technical University, Bhilai (CSVTU Bhilai)
  Gati Shakti Vishwavidyalaya, Vadodara
  Ghani Khan Choudhary Institute of Engineering and Technology, Malda, West Bengal
  Gurukula Kangri Vishwavidyalaya, Haridwar
  HNB Garhwal University Srinagar (Garhwal)
  INDIAN INSTITUTE OF INFORMATION TECHNOLOGY SENAPATI MANIPUR
  Indian Institute of Carpet Technology, Bhadohi
  Indian Institute of Engineering Science and Technology, Shibpur
  Indian Institute of Handloom Technology(IIHT), Varanasi
  Indian Institute of Handloom Technology, Salem
  Indian institute of infor

In [18]:
# As discussed in Phase 3, we split seat_type into two columns:
# 1. category_base → the base category without PwD suffix
#    (OPEN, OBC-NCL, SC, ST, EWS)
# 2. is_pwd → True if this is a PwD seat, False otherwise
#
# This gives maximum flexibility in analysis
# Example: "How do OPEN category trends look?" 
#          → filter category_base == 'OPEN' regardless of PwD status

df_clean['is_pwd'] = df_clean['seat_type'].str.contains(
    'PwD', 
    case=False,   # case insensitive
    na=False      # treat NaN as False, not error
)

df_clean['category_base'] = (
    df_clean['seat_type']
    .str.replace(r'\s*\(PwD\)', '', regex=True)  # Remove " (PwD)" suffix
    .str.strip()                                   # Clean any remaining spaces
)

print("PwD flag distribution:")
print(df_clean['is_pwd'].value_counts())
print("\nBase category distribution:")
print(df_clean['category_base'].value_counts())
print("\nVerification — seat_type vs category_base:")
print(df_clean[['seat_type', 'category_base', 'is_pwd']].drop_duplicates())

PwD flag distribution:
is_pwd
False    386330
True      46195
Name: count, dtype: int64

Base category distribution:
category_base
OPEN       115916
OBC-NCL     95665
SC          85111
ST          69590
EWS         66243
Name: count, dtype: int64

Verification — seat_type vs category_base:
           seat_type category_base  is_pwd
0               OPEN          OPEN   False
2            OBC-NCL       OBC-NCL   False
4                 SC            SC   False
6                 ST            ST   False
27        OPEN (PwD)          OPEN    True
30     OBC-NCL (PwD)       OBC-NCL    True
142         SC (PwD)            SC    True
1990        ST (PwD)            ST    True
47855            EWS           EWS   False
48021      EWS (PwD)           EWS    True


In [19]:
# Current gender values are very long strings
# "Female-only (including Supernumerary)" is 38 characters
# This makes charts unreadable
# We create a short label column for use in visualizations

gender_map = {
    'Gender-Neutral': 'GN',
    'Female-only (including Supernumerary)': 'FO'
}

df_clean['gender_short'] = df_clean['gender'].map(gender_map)

print("Gender short label mapping:")
print(df_clean[['gender', 'gender_short']].drop_duplicates())
print(f"\nAny unmapped values: {df_clean['gender_short'].isna().sum()}")

Gender short label mapping:
                                  gender gender_short
0                         Gender-Neutral           GN
1  Female-only (including Supernumerary)           FO

Any unmapped values: 0


In [20]:
# This is our cleaning sign-off check
# We verify every issue from our Phase 3 quality table is resolved

print("=" * 55)
print("FINAL DATA QUALITY REPORT")
print("=" * 55)

print(f"\nTotal rows               : {len(df_clean):,}")
print(f"Total columns            : {len(df_clean.columns)}")

print(f"\nMissing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

print(f"\nUnique institutes        : {df_clean['institute'].nunique()}")
print(f"Institute types          : {df_clean['institute_type'].unique()}")
print(f"Years covered            : {sorted(df_clean['year'].unique())}")
print(f"Rounds present           : {sorted(df_clean['round'].unique())}")
print(f"Gender values            : {df_clean['gender'].unique()}")

print(f"\nRank range:")
print(f"  Opening Rank  min: {df_clean['opening_rank'].min():,}  max: {df_clean['opening_rank'].max():,}")
print(f"  Closing Rank  min: {df_clean['closing_rank'].min():,}  max: {df_clean['closing_rank'].max():,}")

print(f"\nInvalid ranks (closing < opening): {(df_clean['closing_rank'] < df_clean['opening_rank']).sum()}")
print(f"Special round rows       : {df_clean['is_special_round'].sum():,}")
print(f"PwD rows                 : {df_clean['is_pwd'].sum():,}")

FINAL DATA QUALITY REPORT

Total rows               : 432,525
Total columns            : 14

Missing values:
closing_rank    1
dtype: int64

Unique institutes        : 137
Institute types          : <StringArray>
['IIT', 'NIT', 'GFTI', 'IIIT']
Length: 4, dtype: str
Years covered            : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Rounds present           : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Gender values            : <StringArray>
['Gender-Neutral', 'Female-only (including Supernumerary)']
Length: 2, dtype: str

Rank range:
  Opening Rank  min: 1  max: 1,274,910
  Closing Rank  min: 1.0  max: 1,368,129.0

Invalid ranks (closing < opening): 0
Special round rows       : 14,753
PwD rows                 : 46,195


In [23]:
# The original classify_institute function used exact case matching
# These institutes slipped through due to naming inconsistencies
# We fix them with a case-insensitive contains check

def classify_institute_v2(name):
    name_upper = name.upper()  # Convert to uppercase for comparison
    
    if 'INDIAN INSTITUTE OF TECHNOLOGY' in name_upper:
        return 'IIT'
    elif 'NATIONAL INSTITUTE OF TECHNOLOGY' in name_upper:
        return 'NIT'
    elif 'INDIAN INSTITUTE OF INFORMATION TECHNOLOGY' in name_upper:
        return 'IIIT'
    elif 'INTERNATIONAL INSTITUTE OF INFORMATION TECHNOLOGY' in name_upper:
        return 'IIIT'
    else:
        return 'GFTI'

# Apply the improved classifier
df_clean['institute_type'] = df_clean['institute'].apply(classify_institute_v2)

print("Updated institute type distribution:")
print(df_clean['institute_type'].value_counts())

Updated institute type distribution:
institute_type
NIT     233875
IIT     119011
GFTI     46073
IIIT     33565
Name: count, dtype: int64


In [24]:
# Re-check GFTI list after fix
gfti_institutes_v2 = (
    df_clean[df_clean['institute_type'] == 'GFTI']['institute']
    .unique()
)

print(f"Total GFTI institutes after fix: {len(gfti_institutes_v2)}")
print("\nGFTI institutes now:")
for inst in sorted(gfti_institutes_v2):
    print(f"  {inst}")

Total GFTI institutes after fix: 53

GFTI institutes now:
  Assam University, Silchar
  Birla Institute of Technology, Deoghar Off-Campus
  Birla Institute of Technology, Mesra, Ranchi
  Birla Institute of Technology, Patna Off-Campus
  CU Jharkhand
  Central University of Haryana
  Central University of Jammu
  Central University of Rajasthan, Rajasthan
  Central institute of Technology Kokrajar, Assam
  Chhattisgarh Swami Vivekanada Technical University, Bhilai (CSVTU Bhilai)
  Gati Shakti Vishwavidyalaya, Vadodara
  Ghani Khan Choudhary Institute of Engineering and Technology, Malda, West Bengal
  Gurukula Kangri Vishwavidyalaya, Haridwar
  HNB Garhwal University Srinagar (Garhwal)
  Indian Institute of Carpet Technology, Bhadohi
  Indian Institute of Engineering Science and Technology, Shibpur
  Indian Institute of Handloom Technology(IIHT), Varanasi
  Indian Institute of Handloom Technology, Salem
  Institute of Chemical Technology, Mumbai: Indian Oil Odisha Campus, Bhubaneswar
  

In [25]:
# We use a direct string replacement targeting this exact institute name
# .replace() on a Series replaces exact full matches
# We map the wrong name to the correct name

name_corrections = {
    'lndian Institute of Food Processing Technology, Thanjavur, Tamil Naidu.' : 
    'Indian Institute of Food Processing Technology, Thanjavur, Tamil Nadu'
}

df_clean['institute'] = df_clean['institute'].replace(name_corrections)

# Verify the fix worked
print("Checking for old wrong name:")
print(df_clean[df_clean['institute'].str.contains('lndian', case=True)].shape[0], "rows found")

print("\nChecking new correct name exists:")
print(df_clean[df_clean['institute'].str.contains('Food Processing Technology', case=False)]['institute'].unique())

Checking for old wrong name:
0 rows found

Checking new correct name exists:
<StringArray>
['Indian Institute of Food Processing Technology, Thanjavur, Tamil Nadu']
Length: 1, dtype: str


In [26]:
# Check for names starting with lowercase letter
# Institute names should always start with uppercase
# A lowercase start is a strong signal of a typo

suspicious_names = df_clean[
    df_clean['institute'].str[0].str.islower()
]['institute'].unique()

print(f"Institute names starting with lowercase: {len(suspicious_names)}")
for name in suspicious_names:
    print(f"  '{name}'")

Institute names starting with lowercase: 0


In [27]:
# Always re-save after any additional fixes
output_path = os.path.join('..', 'data', 'processed', 'josaa_clean.csv')
df_clean.to_csv(output_path, index=False)

print(f"Dataset re-saved with latest fixes.")
print(f"Final shape: {df_clean.shape}")

Dataset re-saved with latest fixes.
Final shape: (432524, 14)


In [28]:
# We re-apply the full classification with all edge cases handled
# Using .upper() makes every comparison case-insensitive
# This catches ALL naming variations in one pass

def classify_institute_final(name):
    name_upper = name.upper()
    
    if 'INDIAN INSTITUTE OF TECHNOLOGY' in name_upper:
        return 'IIT'
    
    elif 'NATIONAL INSTITUTE OF TECHNOLOGY' in name_upper:
        return 'NIT'
    
    # This catches: standard naming, ALL CAPS, lowercase variations
    elif 'INDIAN INSTITUTE OF INFORMATION TECHNOLOGY' in name_upper:
        return 'IIIT'
    
    # This catches: International Institute of Information Technology
    elif 'INTERNATIONAL INSTITUTE OF INFORMATION TECHNOLOGY' in name_upper:
        return 'IIIT'
    
    else:
        return 'GFTI'

df_clean['institute_type'] = df_clean['institute'].apply(classify_institute_final)

print("Final institute type distribution:")
print(df_clean['institute_type'].value_counts())

Final institute type distribution:
institute_type
NIT     233875
IIT     119011
GFTI     46073
IIIT     33565
Name: count, dtype: int64


In [29]:
# These two names refer to the exact same institute
# "National Institute of Food Technology Entrepreneurship and Management, Thanjavur"
# "National Institute of Food Technology, Entrepreneurship and Management (NIFTEM) - Thanjavur"
# We standardize to one clean consistent name

niftem_corrections = {
    'National Institute of Food Technology, Entrepreneurship and Management (NIFTEM) - Thanjavur':
    'National Institute of Food Technology Entrepreneurship and Management, Thanjavur'
}

df_clean['institute'] = df_clean['institute'].replace(niftem_corrections)

# Verify
niftem_check = df_clean[
    df_clean['institute'].str.contains('Food Technology', case=False)
]['institute'].unique()

print("NIFTEM institute names after fix:")
for name in niftem_check:
    print(f"  {name}")

NIFTEM institute names after fix:
  National Institute of Food Technology Entrepreneurship and Management, Sonepat, Haryana
  National Institute of Food Technology Entrepreneurship and Management, Thanjavur
  National Institute of Food Technology Entrepreneurship and Management, Kundli


In [30]:
gfti_final = (
    df_clean[df_clean['institute_type'] == 'GFTI']['institute']
    .unique()
)

print(f"Total GFTI institutes: {len(gfti_final)}")
print("\nFinal GFTI list:")
for inst in sorted(gfti_final):
    print(f"  {inst}")

Total GFTI institutes: 52

Final GFTI list:
  Assam University, Silchar
  Birla Institute of Technology, Deoghar Off-Campus
  Birla Institute of Technology, Mesra, Ranchi
  Birla Institute of Technology, Patna Off-Campus
  CU Jharkhand
  Central University of Haryana
  Central University of Jammu
  Central University of Rajasthan, Rajasthan
  Central institute of Technology Kokrajar, Assam
  Chhattisgarh Swami Vivekanada Technical University, Bhilai (CSVTU Bhilai)
  Gati Shakti Vishwavidyalaya, Vadodara
  Ghani Khan Choudhary Institute of Engineering and Technology, Malda, West Bengal
  Gurukula Kangri Vishwavidyalaya, Haridwar
  HNB Garhwal University Srinagar (Garhwal)
  Indian Institute of Carpet Technology, Bhadohi
  Indian Institute of Engineering Science and Technology, Shibpur
  Indian Institute of Food Processing Technology, Thanjavur, Tamil Nadu
  Indian Institute of Handloom Technology(IIHT), Varanasi
  Indian Institute of Handloom Technology, Salem
  Institute of Chemical Te

In [31]:
print("Institute type counts:")
print(df_clean['institute_type'].value_counts())

print(f"\nUnique institutes per type:")
for itype in ['IIT', 'NIT', 'IIIT', 'GFTI']:
    count = df_clean[df_clean['institute_type'] == itype]['institute'].nunique()
    print(f"  {itype}: {count} institutes")

Institute type counts:
institute_type
NIT     233875
IIT     119011
GFTI     46073
IIIT     33565
Name: count, dtype: int64

Unique institutes per type:
  IIT: 23 institutes
  NIT: 31 institutes
  IIIT: 30 institutes
  GFTI: 52 institutes


In [33]:
output_path = os.path.join('..', 'data', 'processed', 'josaa_clean.csv')
df_clean.to_csv(output_path, index=False)

print(f"Final clean dataset saved.")
print(f"Shape: {df_clean.shape}")
print(f"\nColumn list:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i}. {col}")

Final clean dataset saved.
Shape: (432524, 14)

Column list:
  1. institute
  2. program
  3. quota
  4. seat_type
  5. gender
  6. opening_rank
  7. closing_rank
  8. round
  9. year
  10. is_special_round
  11. institute_type
  12. is_pwd
  13. category_base
  14. gender_short


In [34]:
# Fix lowercase 'i' in Central institute of Technology Kokrajar
cit_correction = {
    'Central institute of Technology Kokrajar, Assam':
    'Central Institute of Technology Kokrajar, Assam'
}

df_clean['institute'] = df_clean['institute'].replace(cit_correction)

# Verify
print("Verification:")
print(df_clean[df_clean['institute'].str.contains('Kokrajar')]['institute'].unique())

Verification:
<StringArray>
['Central Institute of Technology Kokrajar, Assam']
Length: 1, dtype: str


In [35]:
# One last comprehensive check across all columns
# This is our formal data quality sign-off

print("=" * 60)
print("PHASE 4 FINAL SIGN-OFF REPORT")
print("=" * 60)

print(f"\n SHAPE")
print(f"  Rows    : {len(df_clean):,}")
print(f"  Columns : {len(df_clean.columns)}")

print(f"\n MISSING VALUES")
total_missing = df_clean.isnull().sum().sum()
print(f"  Total missing values : {total_missing}")

print(f"\n INSTITUTE SUMMARY")
for itype in ['IIT', 'NIT', 'IIIT', 'GFTI']:
    n = df_clean[df_clean['institute_type'] == itype]['institute'].nunique()
    print(f"  {itype} : {n} institutes")

print(f"\n YEAR COVERAGE")
print(f"  {sorted(df_clean['year'].unique())}")

print(f"\n RANK INTEGRITY")
invalid = (df_clean['closing_rank'] < df_clean['opening_rank']).sum()
print(f"  Rows where closing < opening : {invalid}")
print(f"  Opening rank range : {df_clean['opening_rank'].min():,} to {df_clean['opening_rank'].max():,}")
print(f"  Closing rank range : {df_clean['closing_rank'].min():,} to {df_clean['closing_rank'].max():,}")

print(f"\n CATEGORICAL INTEGRITY")
print(f"  Unique quotas      : {sorted(df_clean['quota'].unique())}")
print(f"  Unique categories  : {sorted(df_clean['seat_type'].unique())}")
print(f"  Unique genders     : {sorted(df_clean['gender_short'].unique())}")
print(f"  Unique rounds      : {sorted(df_clean['round'].unique())}")

print(f"\n ENGINEERED COLUMNS")
print(f"  is_special_round True  : {df_clean['is_special_round'].sum():,}")
print(f"  is_pwd True            : {df_clean['is_pwd'].sum():,}")
print(f"  category_base values   : {sorted(df_clean['category_base'].unique())}")
print(f"  gender_short values    : {sorted(df_clean['gender_short'].unique())}")

print(f"\n STATUS : READY FOR EDA")

PHASE 4 FINAL SIGN-OFF REPORT

 SHAPE
  Rows    : 432,524
  Columns : 14

 MISSING VALUES
  Total missing values : 0

 INSTITUTE SUMMARY
  IIT : 23 institutes
  NIT : 31 institutes
  IIIT : 30 institutes
  GFTI : 52 institutes

 YEAR COVERAGE
  [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

 RANK INTEGRITY
  Rows where closing < opening : 0
  Opening rank range : 1 to 1,274,910
  Closing rank range : 1 to 1,368,129

 CATEGORICAL INTEGRITY
  Unique quotas      : ['AI', 'AP', 'GO', 'HS', 'JK', 'LA', 'OS']
  Unique categories  : ['EWS', 'EWS (PwD)', 'OBC-NCL', 'OBC-NCL (PwD)', 'OPEN', 'OPEN (PwD)', 'SC', 'SC (PwD)', 'ST', 'ST (PwD)']
  Unique genders     : ['FO', 'GN']
  Unique rounds      : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

 ENGINEERED COLUMNS
  is_special_round True  : 14,753
  is_pwd True            : 46,195
  category_base values   : ['EWS', 'OB

In [36]:
output_path = os.path.join('..', 'data', 'processed', 'josaa_clean.csv')
df_clean.to_csv(output_path, index=False)
print(f"Final dataset saved: {output_path}")
print(f"Shape: {df_clean.shape}")

Final dataset saved: ..\data\processed\josaa_clean.csv
Shape: (432524, 14)
